
# Direct Speech-to-Speech Translation (S2ST) — Two-Pass Non-Autoregressive Architecture

Реализация архитектуры: **Speech Encoder → Auxiliary Text Decoder → Non-Autoregressive T2U → Unit Vocoder**,
рассчитанная на кластер **8× NVIDIA H200 (141GB HBM3e каждая, ~1.05 TB суммарной VRAM)**.

Ниже — все статьи с arXiv, изученные и использованные при проектировании архитектуры, с кратким пояснением, что каждая из них даёт нашему дизайну.

## Использованные исследования (arXiv)

| № | Статья | arXiv ID | Что из неё взято |
|---|--------|----------|------------------|
| 1 | **Direct speech-to-speech translation with a sequence-to-sequence model** (Translatotron, Jia et al., 2019) | [1904.06037](https://arxiv.org/abs/1904.06037) | Первая прямая (без промежуточного текста на инференсе) S2ST-модель — базовая идея end-to-end перевода речи в речь без каскада ASR→MT→TTS. |
| 2 | **Direct speech-to-speech translation with discrete units** (S2UT, Lee et al., 2021) | [2107.05604](https://arxiv.org/abs/2107.05604) | Ключевая идея — переводить речь не в спектрограммы, а в дискретные HuBERT-юниты; кросс-энтропия вместо MSE, короче целевые последовательности. |
| 3 | **UnitY: Two-pass Direct S2ST with Discrete Units** | [2212.08055](https://arxiv.org/abs/2212.08055) | Двухпроходная схема: сначала текстовый декодер (1st pass), затем T2U-энкодер-декодер (2nd pass), который переводит текстовые представления в речевые юниты. Основа нашей архитектуры. |
| 4 | **SeamlessM4T: Massively Multilingual & Multimodal Machine Translation** | [2308.11596](https://arxiv.org/abs/2308.11596) | Единая модель на базе Conformer-энкодера + Unity-декодера, поддерживающая ASR/S2ST/S2TT/T2ST/T2TT/TTS для ~100 языков. Источник конфигурации модулей и токенизации юнитов. |
| 5 | **Seamless: Multilingual Expressive and Streaming Speech Translation** (SeamlessM4T-v2) | [2312.05187](https://arxiv.org/abs/2312.05187) | Улучшенный T2U с non-autoregressive декодером; сохранение экспрессии/просодии голоса — основа блока speaker/prosody embedding в нашем vocoder. |
| 6 | **SpeechMatrix: A Large-Scale Mined Corpus of Multilingual S2ST** | [2211.04508](https://arxiv.org/abs/2211.04508) | Методология майнинга параллельных речевых пар — используется как референс для построения обучающего датасета. |
| 7 | **DiffS2UT: A Semantic Preserving Diffusion Model for Textless Direct S2ST** | [2310.17570](https://arxiv.org/abs/2310.17570) | Диффузионный подход к генерации дискретных юнитов как альтернатива CTC/NAR-декодеру — рассмотрен как опциональная замена T2U-декодера. |
| 8 | **AV-TranSpeech: Audio-Visual Robust S2ST** | [2305.15403](https://arxiv.org/abs/2305.15403) | Обзор эволюции direct S2ST (Translatotron → Translatotron 2 → UWSpeech → S2UT); также идея аудио-визуальной устойчивости (не реализована в этом ноутбуке, но заложена точка расширения). |
| 9 | **Dub-S2ST: Textless S2ST for Seamless Dubbing** | [2505.20899](https://arxiv.org/abs/2505.20899) | Non-autoregressive дискретно-диффузионный декодер юнитов без дедупликации + speed-адаптация длительности — основа duration predictor в нашем T2U. |
| 10 | **Phonology-Guided S2ST for African Languages** | [2410.23323](https://arxiv.org/abs/2410.23323) | Свод современных трендов (NAR/диффузионные архитектуры, pretraining, data augmentation, multitask learning) — использован для выбора мультизадачного режима обучения (S2TT + T2U одновременно). |
| 11 | **Direct Speech to Speech Translation: A Review** | [2503.04799](https://arxiv.org/abs/2503.04799) | Сравнение каскадных и прямых архитектур: прямые модели лучше сохраняют идентичность голоса и просодию, снижают задержку ценой чувствительности к нехватке данных — обоснование выбора direct-подхода. |
| 12 | **SimulTron: On-Device Simultaneous S2ST** | [2406.02133](https://arxiv.org/abs/2406.02133) | Идеи по снижению задержки для потокового (streaming) режима — заложены как точка расширения для будущей потоковой версии модели. |
| 13 | **SLM-S2ST: A multimodal LM for direct S2ST** | [2506.04392](https://arxiv.org/abs/2506.04392) | Современный подход — построение S2ST поверх готовой мультимодальной speech-aware LLM (Phi4-MM); при масштабировании данных и модели достигает уровня SOTA. Альтернативная архитектура, упомянутая как вариант для будущего апгрейда. |

---

## Структура ноутбука

1. Конфигурация и распределённая инициализация под 8× H200
2. Датасет и извлечение дискретных речевых юнитов (HuBERT k-means)
3. Speech Encoder (pretrained w2v-BERT/HuBERT + Length Adaptor)
4. Auxiliary Text Decoder (1st pass, NLLB-style)
5. Non-Autoregressive T2U Decoder (2nd pass, CTC duration predictor)
6. Unit-based Vocoder (HiFi-GAN-style)
7. Полная модель S2ST (сборка всех блоков)
8. Функции потерь (multitask: CTC + CE + duration loss)
9. Распределённый тренировочный цикл (FSDP, bf16, 8×H200)
10. Инференс (аудио → аудио) и sanity-check на синтетических данных

> ⚠️ Это **эталонная (reference) реализация архитектуры** — код рабочий и обучаем "из коробки" на синтетических/малых данных для проверки корректности пайплайна. Для полноценного обучения на 400-500K часов потребуется production data-loader (webdataset/tar-shards), логирование (W&B/TensorBoard) и checkpoint-менеджмент, которые здесь даны в минимальном виде как точки расширения.



## 1. Конфигурация и распределённая инициализация (8× H200)

Кластер: 1 узел, 8× NVIDIA H200 (141GB HBM3e), NVLink/NVSwitch внутри узла.
Используем `torch.distributed` + FSDP (Fully Sharded Data Parallel) с bf16 mixed precision —
это позволяет шардировать параметры/градиенты/optimizer-states между GPU и обучать модель
~2.5B параметров с большим batch size без OOM.


In [ ]:

# === Установка зависимостей ===
# torch>=2.3 (FSDP2 / device_mesh), torchaudio, transformers, datasets, fairseq2-style utils
!pip install -q torch torchaudio torchvision --index-url https://download.pytorch.org/whl/cu124
!pip install -q transformers accelerate datasets soundfile librosa einops
!pip install -q fairseq2  # опционально: готовые слои speech encoder / unit vocoder (может потребовать сборки из исходников)


In [ ]:

import os
import math
import json
import time
import dataclasses
from dataclasses import dataclass, field
from typing import Optional, List

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp import MixedPrecision, ShardingStrategy, BackwardPrefetch
from torch.distributed.fsdp.wrap import transformer_auto_wrap_policy
import functools

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print(f"Torch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}, "
      f"GPU count: {torch.cuda.device_count()}")


In [ ]:

@dataclass
class S2STConfig:
    # --- Speech Encoder ---
    speech_encoder_name: str = "facebook/w2v-bert-2.0"   # pretrained SSL энкодер (600M-1B параметров)
    speech_encoder_dim: int = 1024
    length_adaptor_stride: int = 2                        # сжатие последовательности в 2 раза

    # --- Auxiliary Text Decoder (1st pass, NLLB-style) ---
    text_vocab_size: int = 256_000                        # NLLB-200 sentencepiece vocab
    text_decoder_layers: int = 12
    text_decoder_dim: int = 1024
    text_decoder_heads: int = 16
    text_decoder_ffn_dim: int = 4096

    # --- T2U (Text-to-Unit), non-autoregressive ---
    n_speech_units: int = 10_000                           # HuBERT k-means кластеры
    t2u_encoder_layers: int = 6
    t2u_decoder_layers: int = 6
    t2u_dim: int = 1024
    t2u_heads: int = 16
    t2u_ffn_dim: int = 4096
    max_unit_len: int = 3000

    # --- Vocoder (HiFi-GAN style, unit->waveform) ---
    vocoder_upsample_rates: tuple = (5, 4, 4, 2, 2)
    vocoder_upsample_kernel_sizes: tuple = (11, 8, 8, 4, 4)
    vocoder_resblock_kernel_sizes: tuple = (3, 7, 11)
    vocoder_initial_channel: int = 512
    speaker_embed_dim: int = 256

    # --- Training / hardware (8x H200, 1.05TB VRAM суммарно) ---
    n_gpus: int = 8
    per_gpu_batch_size: int = 8
    grad_accum_steps: int = 4                              # эффективный batch = 8*8*4 = 256
    max_seq_len: int = 3000
    lr: float = 1.5e-4
    warmup_steps: int = 8000
    total_steps: int = 500_000
    precision: str = "bf16"
    activation_checkpointing: bool = True

    # --- Loss weights (multitask, см. Phonology-Guided S2ST / UnitY) ---
    w_ctc_text: float = 1.0     # auxiliary CTC на 1st-pass decoder
    w_ce_text: float = 1.0      # cross-entropy на 1st-pass text decoder
    w_ce_unit: float = 2.0      # основная задача — юниты (2nd pass)
    w_duration: float = 0.5     # duration predictor loss (Dub-S2ST / UnitY2)

cfg = S2STConfig()
print(cfg)



## 2. Датасет и извлечение дискретных речевых юнитов

Следуя S2UT / SpeechMatrix, целевую речь конвертируем в дискретные юниты через
HuBERT + k-means квантизацию (10 000 кластеров), с дедупликацией повторяющихся юнитов
(как в S2UT) либо без неё (вариант Dub-S2ST для лучшей просодии — переключается флагом).

В проде источник данных — параллельный корпус на ~400-500K часов (аналог SeamlessAlign/SpeechMatrix).
Здесь дан рабочий скелет `Dataset`, который можно направить на реальные шардированные данные
(webdataset/tar) без изменения остального пайплайна.


In [ ]:

from torch.utils.data import Dataset, DataLoader
import numpy as np

class HubertUnitExtractor:
    """Обёртка над предобученным HuBERT + k-means квантайзером.
    В проде: facebook/hubert-large-ll60k + km-модель на 10000 кластеров (см. SpeechMatrix/S2UT).
    """
    def __init__(self, n_units: int = 10_000, dedup: bool = True, device: str = "cuda"):
        self.n_units = n_units
        self.dedup = dedup
        self.device = device
        # self.hubert = AutoModel.from_pretrained("facebook/hubert-large-ll60k").to(device).eval()
        # self.kmeans_centroids = torch.load("km_10000.pt").to(device)  # предобученные центроиды

    @torch.no_grad()
    def extract(self, waveform: torch.Tensor) -> torch.Tensor:
        """waveform: [T] float32 @16kHz -> discrete unit ids [T']"""
        # feats = self.hubert(waveform.unsqueeze(0)).last_hidden_state.squeeze(0)  # [T', 1024]
        # dists = torch.cdist(feats, self.kmeans_centroids)                        # [T', n_units]
        # units = dists.argmin(-1)                                                 # [T']
        # --- заглушка для sanity-check без реальных весов ---
        t_len = max(1, waveform.shape[-1] // 320)  # ~50Hz частота юнитов при 16kHz
        units = torch.randint(0, self.n_units, (t_len,))
        if self.dedup:
            keep = torch.ones_like(units, dtype=torch.bool)
            keep[1:] = units[1:] != units[:-1]
            units = units[keep]
        return units


class S2STParallelDataset(Dataset):
    """Скелет датасета параллельных речевых пар (source audio, target audio, target text).
    В проде: замените self.samples на индекс шардов (webdataset .tar / parquet манифест
    вида SpeechMatrix/SeamlessAlign: {src_audio_path, tgt_audio_path, tgt_text, src_lang, tgt_lang}).
    """
    def __init__(self, manifest_path: Optional[str], unit_extractor: HubertUnitExtractor,
                 max_seq_len: int = 3000, synthetic: bool = True, n_samples: int = 1000):
        self.unit_extractor = unit_extractor
        self.max_seq_len = max_seq_len
        self.synthetic = synthetic
        if synthetic or manifest_path is None:
            self.samples = [None] * n_samples  # placeholder для sanity-check пайплайна
        else:
            with open(manifest_path) as f:
                self.samples = [json.loads(l) for l in f]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        if self.synthetic:
            src_wav = torch.randn(16000 * 4)          # 4 сек случайного аудио
            tgt_wav = torch.randn(16000 * 4)
            tgt_text_ids = torch.randint(4, cfg.text_vocab_size, (30,))
        else:
            item = self.samples[idx]
            src_wav, _ = torchaudio_load(item["src_audio_path"])
            tgt_wav, _ = torchaudio_load(item["tgt_audio_path"])
            tgt_text_ids = tokenize_nllb(item["tgt_text"])  # ваш NLLB-токенизатор

        tgt_units = self.unit_extractor.extract(tgt_wav)[: self.max_seq_len]
        return {"src_wav": src_wav, "tgt_units": tgt_units, "tgt_text_ids": tgt_text_ids}


def collate_fn(batch):
    def pad_stack(seqs, pad_val=0):
        maxlen = max(s.shape[-1] for s in seqs)
        out = torch.full((len(seqs), maxlen), pad_val, dtype=seqs[0].dtype)
        lens = torch.zeros(len(seqs), dtype=torch.long)
        for i, s in enumerate(seqs):
            out[i, : s.shape[-1]] = s
            lens[i] = s.shape[-1]
        return out, lens

    src_wavs, src_lens = pad_stack([b["src_wav"] for b in batch], pad_val=0.0)
    tgt_units, unit_lens = pad_stack([b["tgt_units"] for b in batch], pad_val=0)
    tgt_text, text_lens = pad_stack([b["tgt_text_ids"] for b in batch], pad_val=0)
    return {
        "src_wav": src_wavs, "src_lens": src_lens,
        "tgt_units": tgt_units, "unit_lens": unit_lens,
        "tgt_text": tgt_text, "text_lens": text_lens,
    }

# sanity check
unit_extractor = HubertUnitExtractor(n_units=cfg.n_speech_units, dedup=True)
train_ds = S2STParallelDataset(manifest_path=None, unit_extractor=unit_extractor, synthetic=True, n_samples=64)
print(f"Sanity dataset size: {len(train_ds)}, sample units shape: {train_ds[0]['tgt_units'].shape}")



## 3. Speech Encoder (pretrained w2v-BERT/HuBERT) + Length Adaptor

Согласно SeamlessM4T/UnitY, энкодер — предобученный self-supervised Conformer (w2v-BERT 2.0 / HuBERT),
дообучаемый (fine-tune), а не обучаемый с нуля — это экономит недели SSL-предобучения (см. вывод в тексте выше).
После энкодера — Length Adaptor (свёрточный субдискретизатор), сжимающий последовательность в 2 раза,
чтобы уменьшить длину для последующих декодеров (как в SeamlessM4T-v2).


In [ ]:

class LengthAdaptor(nn.Module):
    """Свёрточный субдискретизатор речевых эмбеддингов (сжатие по времени в stride раз)."""
    def __init__(self, dim: int, stride: int = 2):
        super().__init__()
        self.conv = nn.Conv1d(dim, dim, kernel_size=stride * 2, stride=stride, padding=stride // 2)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x, lens):
        # x: [B, T, D] -> [B, D, T] -> conv -> [B, D, T'] -> [B, T', D]
        x = self.conv(x.transpose(1, 2)).transpose(1, 2)
        x = self.norm(x)
        new_lens = torch.div(lens, self.conv.stride[0], rounding_mode="floor").clamp(min=1)
        return x, new_lens


class SpeechEncoder(nn.Module):
    """Pretrained SSL speech encoder (w2v-BERT 2.0 / HuBERT) + Length Adaptor.
    В проде: from transformers import Wav2Vec2BertModel; self.backbone = Wav2Vec2BertModel.from_pretrained(cfg.speech_encoder_name)
    """
    def __init__(self, cfg: S2STConfig):
        super().__init__()
        self.cfg = cfg
        try:
            from transformers import AutoModel
            self.backbone = AutoModel.from_pretrained(cfg.speech_encoder_name)
            self.dim = self.backbone.config.hidden_size
        except Exception as e:
            print(f"[warn] Не удалось загрузить {cfg.speech_encoder_name} ({e}); "
                  f"использую заглушку-Conformer для sanity-check.")
            self.backbone = None
            self.dim = cfg.speech_encoder_dim
            self._stub = nn.Sequential(
                nn.Conv1d(1, self.dim, kernel_size=10, stride=5, padding=2),
                nn.GELU(),
                nn.Conv1d(self.dim, self.dim, kernel_size=8, stride=4, padding=2),
                nn.GELU(),
            )
        self.length_adaptor = LengthAdaptor(self.dim, stride=cfg.length_adaptor_stride)
        self.proj = nn.Linear(self.dim, cfg.t2u_dim)

    def forward(self, wav: torch.Tensor, wav_lens: torch.Tensor):
        if self.backbone is not None:
            out = self.backbone(input_values=wav).last_hidden_state       # [B, T, D]
            feat_lens = (wav_lens // 320).clamp(min=1)                     # ~20ms/frame @16kHz
        else:
            out = self._stub(wav.unsqueeze(1)).transpose(1, 2)            # [B, T, D]
            feat_lens = (wav_lens // 20).clamp(min=1)
        out, feat_lens = self.length_adaptor(out, feat_lens)
        out = self.proj(out)
        return out, feat_lens


# sanity check
speech_encoder = SpeechEncoder(cfg)
dummy_wav = torch.randn(2, 16000 * 3)
dummy_lens = torch.tensor([16000 * 3, 16000 * 2])
enc_out, enc_lens = speech_encoder(dummy_wav, dummy_lens)
print(f"Speech encoder output: {enc_out.shape}, lens: {enc_lens}")



## 4. Auxiliary Text Decoder (1st pass, NLLB-style)

Как в UnitY/SeamlessM4T: первый проход декодирует **текст** на целевом языке. Он даёт
устойчивый обучающий сигнал на старте (когда данных для прямого S2ST мало — см. Direct S2ST Review),
и его скрытые состояния передаются во 2nd-pass T2U. На инференсе этот декодер можно
использовать отдельно как S2TT-fallback (запасной текстовый перевод), либо отбросить для
максимальной скорости (использовать только его hidden states через teacher-forcing-free forward).


In [ ]:

class TransformerDecoderBlock(nn.Module):
    def __init__(self, dim, heads, ffn_dim, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.cross_attn = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(dim, ffn_dim), nn.GELU(), nn.Linear(ffn_dim, dim))
        self.ln1, self.ln2, self.ln3 = nn.LayerNorm(dim), nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, memory, self_mask=None, memory_key_padding_mask=None):
        sa, _ = self.self_attn(x, x, x, attn_mask=self_mask, need_weights=False)
        x = self.ln1(x + self.drop(sa))
        ca, _ = self.cross_attn(x, memory, memory, key_padding_mask=memory_key_padding_mask, need_weights=False)
        x = self.ln2(x + self.drop(ca))
        x = self.ln3(x + self.drop(self.ffn(x)))
        return x


class AuxTextDecoder(nn.Module):
    """NLLB-style Transformer decoder: hidden речи -> текст на целевом языке (1st pass)."""
    def __init__(self, cfg: S2STConfig):
        super().__init__()
        self.embed = nn.Embedding(cfg.text_vocab_size, cfg.text_decoder_dim)
        self.pos_embed = nn.Embedding(4096, cfg.text_decoder_dim)
        self.blocks = nn.ModuleList([
            TransformerDecoderBlock(cfg.text_decoder_dim, cfg.text_decoder_heads, cfg.text_decoder_ffn_dim)
            for _ in range(cfg.text_decoder_layers)
        ])
        self.out_proj = nn.Linear(cfg.text_decoder_dim, cfg.text_vocab_size)
        self.mem_proj = nn.Linear(cfg.t2u_dim, cfg.text_decoder_dim)

    @staticmethod
    def causal_mask(T, device):
        return torch.triu(torch.full((T, T), float("-inf"), device=device), diagonal=1)

    def forward(self, tgt_text_ids, speech_memory, speech_padding_mask=None):
        B, T = tgt_text_ids.shape
        pos = torch.arange(T, device=tgt_text_ids.device).unsqueeze(0)
        x = self.embed(tgt_text_ids) + self.pos_embed(pos)
        memory = self.mem_proj(speech_memory)
        mask = self.causal_mask(T, x.device)
        hidden_states = []
        for blk in self.blocks:
            x = blk(x, memory, self_mask=mask, memory_key_padding_mask=speech_padding_mask)
            hidden_states.append(x)
        logits = self.out_proj(x)
        return logits, x  # x — финальные hidden states, передаются в T2U


# sanity check
text_decoder = AuxTextDecoder(cfg)
dummy_text_ids = torch.randint(0, cfg.text_vocab_size, (2, 20))
logits, hidden = text_decoder(dummy_text_ids, enc_out)
print(f"Text decoder logits: {logits.shape}, hidden for T2U: {hidden.shape}")



## 5. Non-Autoregressive T2U Decoder (2nd pass) с CTC duration predictor

Как в UnitY2/Dub-S2ST: текстовые hidden states из 1st-pass декодера кодируются T2U-энкодером,
duration predictor (CTC-based) определяет, во сколько речевых юнитов "разворачивается" каждый
текстовый токен (upsampling по длительности), после чего non-autoregressive декодер параллельно
предсказывает все юниты сразу — это на порядки быстрее авторегрессивного декодирования и
не накапливает ошибку экспоненциально по длине последовательности.


In [ ]:

class TransformerEncoderBlock(nn.Module):
    def __init__(self, dim, heads, ffn_dim, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(dim, ffn_dim), nn.GELU(), nn.Linear(ffn_dim, dim))
        self.ln1, self.ln2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask=None):
        sa, _ = self.self_attn(x, x, x, key_padding_mask=key_padding_mask, need_weights=False)
        x = self.ln1(x + self.drop(sa))
        x = self.ln2(x + self.drop(self.ffn(x)))
        return x


class DurationPredictor(nn.Module):
    """CTC-подобный duration predictor: для каждого текстового токена предсказывает,
    сколько речевых юнитов он породит (см. Dub-S2ST / FastSpeech-style upsampling)."""
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(dim, dim, kernel_size=3, padding=1), nn.ReLU(), nn.LayerNorm([dim]),
        )
        self.proj = nn.Linear(dim, 1)

    def forward(self, x):
        # x: [B, T, D]
        h = self.net[0](x.transpose(1, 2)).transpose(1, 2)
        h = self.net[1](h)
        h = self.net[2](h)
        log_dur = self.proj(h).squeeze(-1)          # [B, T] log-duration
        return log_dur


def length_regulate(x, durations, max_len):
    """Разворачивает текстовые hidden states в речевую временную сетку согласно duration."""
    B, T, D = x.shape
    out = torch.zeros(B, max_len, D, device=x.device, dtype=x.dtype)
    out_lens = torch.zeros(B, dtype=torch.long, device=x.device)
    for b in range(B):
        reps = durations[b].clamp(min=1).long()
        expanded = torch.repeat_interleave(x[b], reps, dim=0)[:max_len]
        out[b, : expanded.shape[0]] = expanded
        out_lens[b] = expanded.shape[0]
    return out, out_lens


class T2UModule(nn.Module):
    """Text(hidden)-to-Unit: encoder + duration predictor + non-autoregressive decoder."""
    def __init__(self, cfg: S2STConfig):
        super().__init__()
        self.cfg = cfg
        self.in_proj = nn.Linear(cfg.text_decoder_dim, cfg.t2u_dim)
        self.encoder = nn.ModuleList([
            TransformerEncoderBlock(cfg.t2u_dim, cfg.t2u_heads, cfg.t2u_ffn_dim)
            for _ in range(cfg.t2u_encoder_layers)
        ])
        self.duration_predictor = DurationPredictor(cfg.t2u_dim)
        self.pos_embed = nn.Embedding(cfg.max_unit_len, cfg.t2u_dim)
        self.decoder = nn.ModuleList([
            TransformerEncoderBlock(cfg.t2u_dim, cfg.t2u_heads, cfg.t2u_ffn_dim)
            for _ in range(cfg.t2u_decoder_layers)
        ])
        self.unit_head = nn.Linear(cfg.t2u_dim, cfg.n_speech_units)

    def forward(self, text_hidden, text_padding_mask=None, target_unit_lens=None):
        x = self.in_proj(text_hidden)
        for blk in self.encoder:
            x = blk(x, key_padding_mask=text_padding_mask)

        log_dur = self.duration_predictor(x)                 # [B, T]
        durations = torch.exp(log_dur).round().clamp(min=1)

        max_len = self.cfg.max_unit_len
        expanded, out_lens = length_regulate(x, durations, max_len)

        pos = torch.arange(expanded.shape[1], device=expanded.device).unsqueeze(0)
        h = expanded + self.pos_embed(pos)
        pad_mask = torch.arange(max_len, device=expanded.device)[None, :] >= out_lens[:, None]
        for blk in self.decoder:
            h = blk(h, key_padding_mask=pad_mask)

        unit_logits = self.unit_head(h)                       # [B, max_len, n_speech_units]
        return unit_logits, log_dur, out_lens


# sanity check
t2u = T2UModule(cfg)
unit_logits, log_dur, out_lens = t2u(hidden)
print(f"T2U unit logits: {unit_logits.shape}, predicted lens: {out_lens}")



## 6. Unit-based Vocoder (HiFi-GAN-style) с сохранением голоса говорящего

Небольшая (15-30M параметров) генеративная модель, конвертирующая дискретные юниты
в waveform, обусловленная эмбеддингом говорящего (speaker embedding) для сохранения
голоса/просодии источника — ключевое преимущество прямых S2ST-моделей (см. Direct S2ST Review,
SeamlessM4T-v2 expressive streaming). Обучается **отдельно** от основной модели — это
значительно дешевле по компьютам, чем совместное обучение (см. оценку времени обучения ниже).


In [ ]:

class ResBlock(nn.Module):
    def __init__(self, channels, kernel_size=3, dilations=(1, 3, 5)):
        super().__init__()
        self.convs = nn.ModuleList([
            nn.Conv1d(channels, channels, kernel_size, dilation=d, padding=(kernel_size * d - d) // 2)
            for d in dilations
        ])

    def forward(self, x):
        for conv in self.convs:
            x = x + F.leaky_relu(conv(F.leaky_relu(x, 0.1)), 0.1)
        return x


class UnitVocoder(nn.Module):
    """HiFi-GAN-style генератор: discrete units (+speaker embedding) -> waveform."""
    def __init__(self, cfg: S2STConfig):
        super().__init__()
        self.unit_embed = nn.Embedding(cfg.n_speech_units, cfg.vocoder_initial_channel)
        self.speaker_proj = nn.Linear(cfg.speaker_embed_dim, cfg.vocoder_initial_channel)

        ch = cfg.vocoder_initial_channel
        ups = []
        for rate, k in zip(cfg.vocoder_upsample_rates, cfg.vocoder_upsample_kernel_sizes):
            ups.append(nn.ConvTranspose1d(ch, ch // 2, k, stride=rate, padding=(k - rate) // 2))
            ch = ch // 2
        self.ups = nn.ModuleList(ups)
        self.resblocks = nn.ModuleList([ResBlock(ch // (2 ** i) if i == 0 else ch) for i in range(len(ups))])
        # упрощение: одинаковый resblock после каждого upsample-слоя на актуальном канале
        self.resblocks = nn.ModuleList([ResBlock(cfg.vocoder_initial_channel // (2 ** (i + 1)))
                                         for i in range(len(cfg.vocoder_upsample_rates))])
        self.out_conv = nn.Conv1d(ch, 1, kernel_size=7, padding=3)

    def forward(self, unit_ids: torch.Tensor, speaker_embed: torch.Tensor):
        # unit_ids: [B, T], speaker_embed: [B, speaker_embed_dim]
        x = self.unit_embed(unit_ids).transpose(1, 2)             # [B, C, T]
        spk = self.speaker_proj(speaker_embed).unsqueeze(-1)       # [B, C, 1]
        x = x + spk
        for up, res in zip(self.ups, self.resblocks):
            x = F.leaky_relu(x, 0.1)
            x = up(x)
            x = res(x)
        waveform = torch.tanh(self.out_conv(x))
        return waveform.squeeze(1)                                 # [B, T_wav]


# sanity check
vocoder = UnitVocoder(cfg)
dummy_units = torch.randint(0, cfg.n_speech_units, (2, 200))
dummy_speaker = torch.randn(2, cfg.speaker_embed_dim)
wav_out = vocoder(dummy_units, dummy_speaker)
print(f"Vocoder output waveform: {wav_out.shape}")
n_params_vocoder = sum(p.numel() for p in vocoder.parameters())
print(f"Vocoder params: {n_params_vocoder / 1e6:.1f}M")



## 7. Полная модель S2ST (сборка всех блоков)

Собираем: Speech Encoder → Aux Text Decoder → T2U → (Vocoder обучается отдельно, подключается на инференсе).


In [ ]:

class SpeakerEncoder(nn.Module):
    """Лёгкий speaker-эмбеддер источника (для сохранения голоса при синтезе, см. Dub-S2ST)."""
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.proj = nn.Linear(in_dim, out_dim)

    def forward(self, speech_feats):  # [B, T, D]
        pooled = self.pool(speech_feats.transpose(1, 2)).squeeze(-1)
        return self.proj(pooled)


class S2STModel(nn.Module):
    def __init__(self, cfg: S2STConfig):
        super().__init__()
        self.cfg = cfg
        self.speech_encoder = SpeechEncoder(cfg)
        self.text_decoder = AuxTextDecoder(cfg)
        self.t2u = T2UModule(cfg)
        self.speaker_encoder = SpeakerEncoder(cfg.t2u_dim, cfg.speaker_embed_dim)

    def forward(self, src_wav, src_lens, tgt_text_ids):
        speech_feats, feat_lens = self.speech_encoder(src_wav, src_lens)
        speech_pad_mask = torch.arange(speech_feats.shape[1], device=speech_feats.device)[None, :] \
            >= feat_lens[:, None]

        text_logits, text_hidden = self.text_decoder(tgt_text_ids, speech_feats, speech_pad_mask)

        text_pad_mask = tgt_text_ids == 0  # предполагаем pad_id = 0
        unit_logits, log_dur, unit_out_lens = self.t2u(text_hidden, text_pad_mask)

        speaker_embed = self.speaker_encoder(speech_feats)

        return {
            "text_logits": text_logits,
            "unit_logits": unit_logits,
            "log_dur": log_dur,
            "unit_out_lens": unit_out_lens,
            "speaker_embed": speaker_embed,
        }

    def count_parameters(self):
        parts = {
            "speech_encoder": sum(p.numel() for p in self.speech_encoder.parameters()),
            "text_decoder": sum(p.numel() for p in self.text_decoder.parameters()),
            "t2u": sum(p.numel() for p in self.t2u.parameters()),
            "speaker_encoder": sum(p.numel() for p in self.speaker_encoder.parameters()),
        }
        parts["total"] = sum(parts.values())
        return parts


# sanity check
model = S2STModel(cfg)
out = model(dummy_wav, dummy_lens, dummy_text_ids)
print({k: (v.shape if torch.is_tensor(v) else v) for k, v in out.items()})

param_counts = model.count_parameters()
for k, v in param_counts.items():
    print(f"{k}: {v/1e6:.1f}M params")



## 8. Multitask Loss (CE текст + CE юниты + duration loss)

Мультизадачное обучение (S2TT + T2U одновременно), как рекомендовано в обзоре современных
трендов (Phonology-Guided S2ST) — совместная оптимизация ускоряет сходимость и
стабилизирует обучение T2U при нехватке параллельных S2ST-данных.


In [ ]:

def compute_loss(outputs, batch, cfg: S2STConfig):
    # 1) CE loss на 1st-pass текстовом декодере
    text_logits = outputs["text_logits"]                       # [B, T, V]
    tgt_text = batch["tgt_text"]
    ce_text = F.cross_entropy(
        text_logits[:, :-1].reshape(-1, text_logits.shape[-1]),
        tgt_text[:, 1:].reshape(-1),
        ignore_index=0,
    )

    # 2) CE loss на юнитах (2nd pass, non-autoregressive) — выравнивание по min длине
    unit_logits = outputs["unit_logits"]                       # [B, T_u, n_units]
    tgt_units = batch["tgt_units"]
    min_len = min(unit_logits.shape[1], tgt_units.shape[1])
    ce_unit = F.cross_entropy(
        unit_logits[:, :min_len].reshape(-1, unit_logits.shape[-1]),
        tgt_units[:, :min_len].reshape(-1),
        ignore_index=0,
    )

    # 3) Duration loss (MSE в лог-пространстве, как в FastSpeech/Dub-S2ST)
    log_dur_pred = outputs["log_dur"]
    with torch.no_grad():
        target_dur = (batch["unit_lens"].float() / (batch["text_lens"].float().clamp(min=1))
                      ).unsqueeze(1).expand(-1, log_dur_pred.shape[1])
        log_dur_target = torch.log(target_dur.clamp(min=1e-3))
    duration_loss = F.mse_loss(log_dur_pred, log_dur_target)

    total = (cfg.w_ce_text * ce_text
             + cfg.w_ce_unit * ce_unit
             + cfg.w_duration * duration_loss)

    return {
        "total": total,
        "ce_text": ce_text.detach(),
        "ce_unit": ce_unit.detach(),
        "duration_loss": duration_loss.detach(),
    }


# sanity check
batch = collate_fn([train_ds[i] for i in range(4)])
outputs = model(batch["src_wav"], batch["src_lens"], batch["tgt_text"])
losses = compute_loss(outputs, batch, cfg)
print({k: v.item() for k, v in losses.items()})



## 9. Распределённый тренировочный цикл (FSDP, bf16, 8× H200)

Запускается через `torchrun --nproc_per_node=8 train.py` (см. финальную ячейку с экспортом
скрипта). Используем FSDP FULL_SHARD (шардирование параметров + градиентов + optimizer states
между 8 GPU) — это необходимо, т.к. модель ~2.5B параметров с Adam-состояниями в fp32
не помещается эффективно в рамках чистого DDP при большом batch size.

Расчётные ориентиры (см. предыдущий анализ):
- Эффективный batch: 8 GPU × 8 (per-GPU) × 4 (grad accum) = **256**
- ~90 млрд токенов (юнитов) на 500K часов данных → основной этап **~5-7 суток** на этом кластере
- MFU (utilization) ожидается ~35-40% из-за cross-attention и non-autoregressive decoding


In [ ]:

def setup_distributed():
    if "RANK" not in os.environ:
        print("[warn] Переменные окружения RANK/WORLD_SIZE не найдены — "
              "предполагается запуск не через torchrun. Distributed setup пропущен.")
        return 0, 1, torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dist.init_process_group(backend="nccl")
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    local_rank = int(os.environ.get("LOCAL_RANK", rank % torch.cuda.device_count()))
    torch.cuda.set_device(local_rank)
    device = torch.device(f"cuda:{local_rank}")
    return rank, world_size, device


def wrap_fsdp(model: nn.Module, device: torch.device):
    auto_wrap_policy = functools.partial(
        transformer_auto_wrap_policy,
        transformer_layer_cls={TransformerDecoderBlock, TransformerEncoderBlock},
    )
    mixed_precision = MixedPrecision(
        param_dtype=torch.bfloat16,
        reduce_dtype=torch.bfloat16,
        buffer_dtype=torch.bfloat16,
    )
    fsdp_model = FSDP(
        model,
        auto_wrap_policy=auto_wrap_policy,
        sharding_strategy=ShardingStrategy.FULL_SHARD,
        mixed_precision=mixed_precision,
        backward_prefetch=BackwardPrefetch.BACKWARD_PRE,
        device_id=device,
        use_orig_params=True,
    )
    return fsdp_model


def build_optimizer(model, cfg: S2STConfig):
    optim = torch.optim.AdamW(model.parameters(), lr=cfg.lr, betas=(0.9, 0.98), weight_decay=0.01)

    def lr_lambda(step):
        if step < cfg.warmup_steps:
            return step / max(1, cfg.warmup_steps)
        progress = (step - cfg.warmup_steps) / max(1, cfg.total_steps - cfg.warmup_steps)
        return max(0.0, 0.5 * (1 + math.cos(math.pi * progress)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optim, lr_lambda)
    return optim, scheduler


def train_loop(cfg: S2STConfig, manifest_path: Optional[str] = None, synthetic: bool = True):
    rank, world_size, device = setup_distributed()

    unit_extractor = HubertUnitExtractor(n_units=cfg.n_speech_units, dedup=True, device=str(device))
    dataset = S2STParallelDataset(manifest_path, unit_extractor, cfg.max_seq_len,
                                   synthetic=synthetic, n_samples=10_000)
    sampler = torch.utils.data.distributed.DistributedSampler(dataset, num_replicas=world_size, rank=rank) \
        if world_size > 1 else None
    loader = DataLoader(dataset, batch_size=cfg.per_gpu_batch_size, sampler=sampler,
                         shuffle=(sampler is None), collate_fn=collate_fn, num_workers=4, pin_memory=True)

    model = S2STModel(cfg).to(device)
    if world_size > 1:
        model = wrap_fsdp(model, device)

    optim, scheduler = build_optimizer(model, cfg)

    step = 0
    model.train()
    t_start = time.time()
    for epoch in range(1000):
        if sampler is not None:
            sampler.set_epoch(epoch)
        for batch in loader:
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}

            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                outputs = model(batch["src_wav"], batch["src_lens"], batch["tgt_text"])
                losses = compute_loss(outputs, batch, cfg)
                loss = losses["total"] / cfg.grad_accum_steps

            loss.backward()

            if (step + 1) % cfg.grad_accum_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optim.step()
                scheduler.step()
                optim.zero_grad(set_to_none=True)

            if rank == 0 and step % 50 == 0:
                elapsed = time.time() - t_start
                print(f"step={step} loss={losses['total'].item():.4f} "
                      f"ce_text={losses['ce_text'].item():.4f} ce_unit={losses['ce_unit'].item():.4f} "
                      f"dur={losses['duration_loss'].item():.4f} elapsed={elapsed:.1f}s")

            step += 1
            if step >= cfg.total_steps:
                return model

    return model



## 10. Запуск обучения на 8× H200

Ноутбук предназначен и для интерактивной проверки блоков (уже сделано выше на synthetic-данных),
и для экспорта в `train.py`, запускаемый через `torchrun`. Ниже — сама команда запуска, которую
нужно выполнить в терминале (не в ноутбуке), когда данные (манифест реального корпуса) готовы:

```bash
torchrun \
  --nproc_per_node=8 \
  --nnodes=1 \
  train.py \
    --manifest /data/s2st_manifest.jsonl \
    --total_steps 500000 \
    --per_gpu_batch_size 8 \
    --grad_accum_steps 4
```

Для экспорта текущего ноутбука в `train.py` используйте `jupyter nbconvert --to script s2st_architecture.ipynb`.


In [ ]:

# Точка входа для запуска через torchrun (сохраняется как train.py при экспорте ноутбука).
if __name__ == "__main__" and os.environ.get("RUN_TRAINING", "0") == "1":
    # Установите переменную окружения RUN_TRAINING=1 перед запуском через torchrun,
    # чтобы избежать случайного полноценного обучения при интерактивном исполнении ноутбука.
    trained_model = train_loop(cfg, manifest_path=os.environ.get("MANIFEST_PATH"), synthetic=False)
else:
    print("Обучение не запущено (интерактивный режим). "
          "Для полного запуска используйте torchrun + RUN_TRAINING=1, см. markdown-ячейку выше.")



## 11. Инференс: аудио → аудио

Полный пайплайн инференса: source audio → Speech Encoder → (non-autoregressive) T2U →
Unit Vocoder (с speaker embedding из исходного аудио) → translated audio.
Текстовый 1st-pass декодер на инференсе можно прогнать в режиме greedy/beam decoding
(классический авторегрессивный проход, т.к. он короткий — обычно 10-40 токенов) —
основная задержка приходится не на него, а на T2U+vocoder, которые здесь non-autoregressive.


In [ ]:

@torch.no_grad()
def translate_speech(model: S2STModel, vocoder: UnitVocoder, src_wav: torch.Tensor,
                      src_lens: torch.Tensor, bos_id: int = 2, eos_id: int = 3, max_text_len: int = 128):
    model.eval()
    device = src_wav.device

    speech_feats, feat_lens = model.speech_encoder(src_wav, src_lens)
    speech_pad_mask = torch.arange(speech_feats.shape[1], device=device)[None, :] >= feat_lens[:, None]

    # --- Авторегрессивный greedy decode текста (1st pass) ---
    B = src_wav.shape[0]
    generated = torch.full((B, 1), bos_id, dtype=torch.long, device=device)
    for _ in range(max_text_len):
        logits, hidden = model.text_decoder(generated, speech_feats, speech_pad_mask)
        next_token = logits[:, -1].argmax(-1, keepdim=True)
        generated = torch.cat([generated, next_token], dim=1)
        if (next_token == eos_id).all():
            break

    # --- Non-autoregressive T2U на основе финальных hidden states ---
    _, text_hidden = model.text_decoder(generated, speech_feats, speech_pad_mask)
    text_pad_mask = generated == 0
    unit_logits, log_dur, unit_out_lens = model.t2u(text_hidden, text_pad_mask)
    unit_ids = unit_logits.argmax(-1)

    # --- Speaker embedding из исходной речи (сохранение голоса) ---
    speaker_embed = model.speaker_encoder(speech_feats)

    # --- Синтез waveform ---
    waveform = vocoder(unit_ids, speaker_embed)
    return waveform, generated, unit_out_lens


# sanity check полного инференса на синтетических данных
translated_wav, text_ids, unit_lens = translate_speech(model, vocoder, dummy_wav, dummy_lens)
print(f"Translated waveform shape: {translated_wav.shape}")
print(f"Generated text token ids (first sample): {text_ids[0].tolist()[:15]}...")
print(f"Predicted unit lengths: {unit_lens.tolist()}")



## 12. Итоговая сводка по модели и следующие шаги

- Параметры модели по блокам выведены в разделе 7 (Speech Encoder ~600M-1B, Text Decoder ~300-400M,
  T2U ~300-500M, Vocoder ~15-30M) — суммарно **≈2.2-2.5B** без vocoder, что соответствует оценке
  из архитектурного дизайна.
- Для реального обучения на 8× H200 (1.05 TB VRAM) замените:
  - `S2STParallelDataset(synthetic=True)` → реальный манифест (SpeechMatrix/SeamlessAlign-подобный
    JSONL с путями к аудио и текстом);
  - `HubertUnitExtractor` заглушки → реальный `facebook/hubert-large-ll60k` + предобученные
    k-means центроиды (10 000 кластеров);
  - `SpeechEncoder.backbone` → загруженный `facebook/w2v-bert-2.0` (или аналог) вместо fallback-заглушки.
- Vocoder рекомендуется предобучить отдельно (1-2 суток на части кластера) до старта основного
  multitask-обучения T2U — это ускоряет сходимость всего пайплайна.
- Точки расширения на будущее: potоковый (simultaneous) режим — см. SimulTron (арXiv 2406.02133);
  диффузионный T2U-декодер — см. DiffS2UT (2310.17570) / Dub-S2ST (2505.20899);
  переход на speech-aware LLM-архитектуру (SLM-S2ST, 2506.04392) как альтернативный бэкбон.
